In [ ]:
import os
os.environ["HF_TOKEN"]="hf_XXXXXXXXX" # Hugging Face API token

### Installing Libraries

In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain-huggingface faiss-cpu tiktoken python-dotenv

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


### Step 1.A : Indexing (Document Ingestion)

In [ ]:
# from youtube_transcript_api import YouTubeTranscriptApi
# from youtube_transcript_api._errors import TranscriptsDisabled

video_id = "SwQhKFMxmDY" #only the video id, not the entire url
try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript_list = fetched_transcript.to_raw_data()  #
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")



In [ ]:
print(fetched_transcript)

In [ ]:
print(transcript_list)

### Step 1.B : Indexing (Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

In [ ]:
chunks[0]

### Step 1.C & 1.D : Indexing (Embedding Creation & Vector Store)

In [ ]:
!pip install --upgrade pillow sentence-transformers transformers faiss-cpu

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

In [ ]:
vectorstore.index_to_docstore_id

In [ ]:
vectorstore.get_by_ids(['5d828406-15a7-4bf7-882a-b92208a010cf'])

### Step 2 : Retrieval

In [ ]:
retreiver = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [ ]:
retreiver

In [ ]:
retreiver.invoke('What is this podcast about?')

### Step 3 : Augmentation

In [ ]:
prompt = PromptTemplate(
    template="""Based on the context below, answer the question directly and concisely, And say that "You dont know" if question is ut of context .

Context: {context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)


In [ ]:
question = "What is this podcast about?"
retreived_docs = retreiver.invoke(question)

In [ ]:
retreived_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retreived_docs)
context_text

In [ ]:
final_prompt = prompt.format(context=context_text, question=question)
final_prompt

### Step 4: Generation

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Using distilgpt2 (small, fast)
model_id = "distilgpt2"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.2,
    do_sample=True,
    device=-1  # CPU
)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
answer = llm.invoke(final_prompt)
answer_only = answer.split("Question: What is this podcast about?")[-1].strip()
print("Question: ", question)
print( answer_only)